In [36]:
import numpy as np
import pandas as pd
import json

np.bool = np.bool_
from docplex.cp.model import *
import networkx as nx

df_name = pd.read_excel('../data/생관3-612-24-007_2024년 4월 일일생산계획 및 업체별 차종현황_Rev.00_24.03.28(확정).xlsx',
                        skiprows=[0, 1, 2], usecols='C', nrows=64)
df_data = pd.read_excel('../data/생관3-612-24-007_2024년 4월 일일생산계획 및 업체별 차종현황_Rev.00_24.03.28(확정).xlsx',
                        skiprows=[0, 1, 2, 3], usecols='F:BE', nrows=64)



In [37]:
def clean_column_names(df):
    columns = df.columns.tolist()
    for i in range(1, len(columns)):  # 첫 번째 열은 처리할 필요 없음
        if isinstance(columns[i], str) and "Unnamed" in columns[i]:
            columns[i] = columns[i - 1]  # 바로 왼쪽 열 이름으로 대체
    df.columns = columns
    return df


In [38]:
df_name = df_name.drop(index=df_name.index[0])
df_name = df_name.reset_index(drop=True)
df_name = df_name.map(lambda x: x.strip() if isinstance(x, str) else x)
df_name = df_name.ffill()
df_data = df_data.drop(index=df_data.index[-1])
df_data = df_data.drop(index=df_data.index[-1])
df_data.index = df_name['세부공정명'].tolist()

# DataFrame 열 이름 정리
df_data = clean_column_names(df_data)

In [39]:
class SubwayCar:
    def __init__(self, name):
        self.name = name
        self.activity_dict = dict()
        self.activity_indices = []
        self.special_activity_indices = []
        self.min_date_index = None
        self.min_date_index_wo_special = None

    def add_activity(self, activity_name, date):
        if activity_name not in self.activity_dict:
            self.activity_dict[activity_name] = date
        else:
            print(f"subwaycar {self.name} activity {activity_name}: {date} is already taken")
            
            
def create_subway_car_dict(df_data):
    subway_car_dict = dict()
    temp = list(dict.fromkeys(df_data.columns))
    for col_idx in range(df_data.shape[1]):
        date = df_data.columns[col_idx]  # 현재 열의 이름

        for row_idx in range(df_data.shape[0]):  # 행 반복
            activity_name = df_data.index[row_idx]  # 현재 행 이름
            car_name = df_data.iloc[row_idx, col_idx]  # 값 가져오기

            if pd.notna(car_name):  # NaN 값 제외
                if car_name not in subway_car_dict:
                    subway_car_dict[car_name] = SubwayCar(name=car_name)
                # subway_car_dict[car_name].add_activity(activity_name, start_date + pd.Timedelta(days=date - 1))
                subway_car_dict[car_name].add_activity(activity_name, temp.index(date) + 1)
    return subway_car_dict

In [40]:
subway_car_dict = create_subway_car_dict(df_data)

In [41]:
subway_car_dict['S036-T2'].activity_dict

{'흡음재 취부': 1,
 '흡음재 씰링/검사/수정': 2,
 '측창취부/T-BOLT 삽입': 2,
 '실내/상하 CABLE HARNESS 취부': 3,
 '실내/상하 배선': 4,
 '실내 배선 작업': 5,
 '리무벌/파티션 프레임 취부': 6,
 '하부덕트검사\n (D+1~4일차 조정작업)': 7,
 '실내 배선 검사': 8,
 'AIR DUCT MODULE 취부': 24,
 'CENTER GRILL 취부': 25,
 'AIR DUCT MODULE 취부 검사': 26}

In [42]:
calendar = {i: (pd.to_datetime('2022-04-01') + pd.Timedelta(days=i)).strftime('%Y-%m-%d')
 for i in range(30)}
calendar

{0: '2022-04-01',
 1: '2022-04-02',
 2: '2022-04-03',
 3: '2022-04-04',
 4: '2022-04-05',
 5: '2022-04-06',
 6: '2022-04-07',
 7: '2022-04-08',
 8: '2022-04-09',
 9: '2022-04-10',
 10: '2022-04-11',
 11: '2022-04-12',
 12: '2022-04-13',
 13: '2022-04-14',
 14: '2022-04-15',
 15: '2022-04-16',
 16: '2022-04-17',
 17: '2022-04-18',
 18: '2022-04-19',
 19: '2022-04-20',
 20: '2022-04-21',
 21: '2022-04-22',
 22: '2022-04-23',
 23: '2022-04-24',
 24: '2022-04-25',
 25: '2022-04-26',
 26: '2022-04-27',
 27: '2022-04-28',
 28: '2022-04-29',
 29: '2022-04-30'}

In [43]:
class Operation:
    def __init__(self, name, order, operation_name):
        # self.name = name
        self.operation_name = operation_name
        self.order = order
        self.type = ['T', 'M', 'TC']
        self.group = 'A'
        self.duration = 1
        self.resource = 1

In [44]:
process_list = list(df_name['세부공정명'])
operation_list = ["operation" + str(i) for i in range(len(process_list))]
operation_dict = {operation_list[i]:Operation(operation_list[i], i, process_list[i]) for i in range(len(process_list))}

In [45]:
class Vehicles:
    def __init__(self, vehicle_type, operation_dict):
        self.type = vehicle_type
        self.operation_dict = operation_dict
        

In [46]:
vehicle_dict = {}
for key, value in subway_car_dict.items():
    if 'TC' in key:
        vehicle_type = 'TC'
    elif 'M' in key:
        vehicle_type = 'M'
    elif 'T' in key:
        vehicle_type = 'T'
    else:
        vehicle_type = 'None'
        print('error')
    vehicle_dict[key] = Vehicles(vehicle_type, {operation_list[process_list.index(a_key)]: a_value for a_key, a_value in value.activity_dict.items()})

In [47]:
vehicle_dict['S036-T2'].__dict__

{'type': 'T',
 'operation_dict': {'operation0': 1,
  'operation1': 2,
  'operation2': 2,
  'operation3': 3,
  'operation4': 4,
  'operation5': 5,
  'operation7': 6,
  'operation9': 7,
  'operation10': 8,
  'operation11': 24,
  'operation12': 25,
  'operation13': 26}}

In [48]:
for_json = {
    "calendar": {
        "length": len(calendar),
        "data": calendar
    },
    "operations": {
        "num_items": len(operation_list),        
        "data": {key: value.__dict__ for key, value in operation_dict.items()}
    },
    "vehicles": {
        "num_items": len(vehicle_dict),
        "data": {key: value.__dict__ for key, value in vehicle_dict.items()}
    }
}
# json_result = json.dumps(for_json, ensure_ascii=False, indent=4)
# print(json_result)

In [49]:
# 파일 경로 설정
file_path = 'schedule_optimization_output.json'

try:
    with open(file_path, 'w', encoding='utf-8') as f:
        # for_json_final 객체를 f 파일 스트림에 JSON 형식으로 씁니다.
        # ensure_ascii=False: 비 ASCII 문자(예: 한글)가 유니코드 이스케이프 없이 그대로 저장되도록 합니다.
        # indent=4: JSON 파일을 사람이 읽기 쉽게 4칸 들여쓰기로 형식화합니다.
        json.dump(for_json, f, ensure_ascii=False, indent=4)
    print(f"JSON 데이터가 '{file_path}' 파일로 성공적으로 저장되었습니다.")
except IOError as e:
    print(f"파일 저장 중 오류가 발생했습니다: {e}")
except TypeError as e:
    # json.dump가 직렬화할 수 없는 객체를 만났을 때 발생 (예: __dict__로 처리 안 된 사용자 객체)
    print(f"JSON 직렬화 중 오류가 발생했습니다: {e}")

JSON 데이터가 'schedule_optimization_output.json' 파일로 성공적으로 저장되었습니다.
